# Реализация пайплайна скачивания новости и записи ее в БД

In [1]:
import datetime

import pandas as pd
import requests

In [2]:
def get_news_for_a_period(start_date: str, end_date: str, countries=None) -> pd.DataFrame:
    if countries is None:
        countries = ['US', 'FR', 'GB', 'EU', 'AU', 'DE', 'JP', 'CN']
        
    start_date += 'T00:00:00'
    end_date += 'T00:00:00'

    url = 'https://economic-calendar.tradingview.com/events'
    headers = {
        'Origin': 'https://in.tradingview.com'
    }
    payload = {
        'from': start_date + '.000Z',
        'to': end_date + '.000Z',
        'countries': ','.join(countries)
    }
    data = requests.get(url, headers=headers, params=payload).json()
    data = pd.DataFrame(data['result'])
    return data

In [3]:
def get_news_for_today(countries=None):
    today = datetime.date.today()
    tomorrow = today + datetime.timedelta(days=1)
    
    today = today.strftime('%Y-%m-%d')
    tomorrow = tomorrow.strftime('%Y-%m-%d')
    
    news = get_news_for_a_period(today, tomorrow, countries)
    return news

In [4]:
def get_news_for_a_week(countries=None):
    today = datetime.date.today()
    next_week = today + datetime.timedelta(days=7)
    
    today = today.strftime('%Y-%m-%d')
    next_week = next_week.strftime('%Y-%m-%d')
    
    news = get_news_for_a_period(today, next_week, countries)
    return news    

In [5]:
news = get_news_for_a_week()

In [6]:
news

,id,title,country,indicator,comment,category,period,referenceDate,source,source_url,...,forecast,actualRaw,previousRaw,forecastRaw,currency,importance,date,ticker,unit,scale
0,401347,S&P Global Manufacturing PMI Final,AU,manufacturing pmi,The S&P Global Australia Manufacturing PMI is ...,bsnss,Jan,2026-01-31T00:00:00Z,S&P Global,https://www.pmi.spglobal.com/public,...,NaN,None,5.160000e+01,NaN,AUD,-1,2026-02-01T22:00:00.000Z,NaN,NaN,NaN
1,393949,BoJ Summary of Opinions,JP,Interest Rate,"In Japan, interest rates are set by the Bank o...",mny,,None,Central Bank,https://www.boj.or.jp,...,NaN,None,NaN,NaN,JPY,0,2026-02-01T23:50:00.000Z,ECONOMICS:JPINTR,NaN,NaN
2,400798,TD-MI Inflation Gauge MoM,AU,MI Inflation Gauge MoM,"In Australia, the Melbourne Institute Monthly ...",prce,Jan,2026-01-31T00:00:00Z,Melbourne Institute,https://melbourneinstitute.unimelb.edu.au/,...,NaN,None,1.000000e+00,NaN,AUD,-1,2026-02-02T00:30:00.000Z,ECONOMICS:AUMIIGMM,%,NaN
3,401032,S&P Global Manufacturing PMI Final,JP,Manufacturing PMI,The S&P Global Japan Manufacturing PMI is a mo...,bsnss,Jan,2026-01-31T00:00:00Z,S&P Global,https://www.pmi.spglobal.com/public,...,51.5,None,5.000000e+01,5.150000e+01,JPY,-1,2026-02-02T00:30:00.000Z,NaN,NaN,NaN
4,403429,ANZ-Indeed Job Ads MoM,AU,Job Advertisements,"In Australia, job advertisements measure the n...",lbr,Jan,2026-01-31T00:00:00Z,ANZ - Indeed Australian Job Ads,https://media.anz.com/,...,NaN,None,-5.000000e-01,NaN,AUD,-1,2026-02-02T00:30:00.000Z,ECONOMICS:AUJA,%,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,395005,Baker Hughes Oil Rig Count,US,Crude Oil Rigs,US Crude Oil Rigs refer to the number of activ...,NaN,Feb/06,2026-02-06T00:00:00Z,Baker Hughes Company,https://www.bakerhughes.com/,...,NaN,None,4.110000e+02,NaN,USD,-1,2026-02-06T18:00:00.000Z,ECONOMICS:USCOR,NaN,NaN
196,395006,Baker Hughes Total Rigs Count,US,Total Rigs,US Total Rigs refer to the number of active US...,NaN,Feb/06,2026-02-06T00:00:00Z,Baker Hughes Company,https://www.bakerhughes.com/,...,NaN,None,5.460000e+02,NaN,USD,-1,2026-02-06T18:00:00.000Z,ECONOMICS:USTRG,NaN,NaN
197,398452,Consumer Credit Change,US,Consumer Credit,"In the United States, Consumer Credit refers t...",cnsm,Dec,2025-12-31T00:00:00Z,Federal Reserve,https://www.federalreserve.gov,...,8.5,None,4.230000e+09,8.500000e+09,USD,-1,2026-02-06T20:00:00.000Z,ECONOMICS:USCCR,$,B
198,397860,Foreign Exchange Reserves,CN,Foreign Exchange Reserves,"In China, Foreign Exchange Reserves are the fo...",mny,Jan,2026-01-31T00:00:00Z,Central Bank,http://www.pbc.gov.cn,...,NaN,None,3.358000e+12,NaN,CNY,-1,2026-02-07T08:00:00.000Z,ECONOMICS:CNFER,$,T


In [7]:
news.columns

Index(['id', 'title', 'country', 'indicator', 'comment', 'category', 'period',
       'referenceDate', 'source', 'source_url', 'actual', 'previous',
       'forecast', 'actualRaw', 'previousRaw', 'forecastRaw', 'currency',
       'importance', 'date', 'ticker', 'unit', 'scale'],
      dtype='object')

## Создаем БД на sql lite

In [8]:
import sqlite3

In [9]:
conn = sqlite3.connect('economic_news.db')

In [10]:
news.to_sql('news', conn, if_exists='replace', index=False)

200

In [11]:
cursor = conn.cursor()

In [20]:
news_db = pd.read_sql_query("SELECT date as dt, title, indicator, country, date(date) as date, strftime('%d', date) as day FROM news", conn)

In [19]:
news_db

,dt,title,indicator,country,date,"strftime('%d', date)"
0,2026-02-01T22:00:00.000Z,S&P Global Manufacturing PMI Final,manufacturing pmi,AU,2026-02-01,01
1,2026-02-01T23:50:00.000Z,BoJ Summary of Opinions,Interest Rate,JP,2026-02-01,01
2,2026-02-02T00:30:00.000Z,TD-MI Inflation Gauge MoM,MI Inflation Gauge MoM,AU,2026-02-02,02
3,2026-02-02T00:30:00.000Z,S&P Global Manufacturing PMI Final,Manufacturing PMI,JP,2026-02-02,02
4,2026-02-02T00:30:00.000Z,ANZ-Indeed Job Ads MoM,Job Advertisements,AU,2026-02-02,02
...,...,...,...,...,...,...
195,2026-02-06T18:00:00.000Z,Baker Hughes Oil Rig Count,Crude Oil Rigs,US,2026-02-06,06
196,2026-02-06T18:00:00.000Z,Baker Hughes Total Rigs Count,Total Rigs,US,2026-02-06,06
197,2026-02-06T20:00:00.000Z,Consumer Credit Change,Consumer Credit,US,2026-02-06,06
198,2026-02-07T08:00:00.000Z,Foreign Exchange Reserves,Foreign Exchange Reserves,CN,2026-02-07,07


In [23]:
news_db = pd.read_sql_query("""
    SELECT 
        date as dt, 
        title, 
        indicator, 
        country, 
        source,
        importance,
        date(date) as date, 
        strftime('%d', date) as day,
        COUNT(*) OVER (PARTITION BY strftime('%d', date)) as news_count_per_day,
        RANK() OVER (PARTITION BY strftime('%d', date) ORDER BY importance DESC) as importance_rank_per_day,
        COUNT(*) 
    FROM news
    """, conn)
news_db

,dt,title,indicator,country,source,importance,date,day,news_count_per_day,importance_rank_per_day
0,2026-02-01T23:50:00.000Z,BoJ Summary of Opinions,Interest Rate,JP,Central Bank,0,2026-02-01,01,2,1
1,2026-02-01T22:00:00.000Z,S&P Global Manufacturing PMI Final,manufacturing pmi,AU,S&P Global,-1,2026-02-01,01,2,2
2,2026-02-02T01:45:00.000Z,RatingDog Manufacturing PMI,Manufacturing PMI,CN,S&P Global,1,2026-02-02,02,33,1
3,2026-02-02T15:00:00.000Z,ISM Manufacturing PMI,Business Confidence,US,Institute for Supply Management,1,2026-02-02,02,33,1
4,2026-02-02T07:00:00.000Z,Nationwide Housing Prices MoM,nationwide housing prices mom,GB,Nationwide Building Society,0,2026-02-02,02,33,3
...,...,...,...,...,...,...,...,...,...,...
195,2026-02-06T18:00:00.000Z,Baker Hughes Oil Rig Count,Crude Oil Rigs,US,Baker Hughes Company,-1,2026-02-06,06,40,16
196,2026-02-06T18:00:00.000Z,Baker Hughes Total Rigs Count,Total Rigs,US,Baker Hughes Company,-1,2026-02-06,06,40,16
197,2026-02-06T20:00:00.000Z,Consumer Credit Change,Consumer Credit,US,Federal Reserve,-1,2026-02-06,06,40,16
198,2026-02-07T08:00:00.000Z,Foreign Exchange Reserves,Foreign Exchange Reserves,CN,Central Bank,-1,2026-02-07,07,1,1
